# Deep Q-Learning (DQL)

Deep Q-Learning combines **Q-Learning** with **Deep Neural Networks** to handle large state spaces.

**Bellman Equation:**
```
Q(s, a) = r + gamma * max_a' Q(s', a')
```

### Key Components
- **Q-Network** — neural net that approximates Q(s,a)
- **Replay Buffer** — stores past (s,a,r,s',done) transitions
- **Target Network** — frozen copy of online network for stable targets
- **Epsilon-Greedy** — exploration vs exploitation


In [ ]:
# Install dependencies if needed
# !pip install gymnasium torch numpy matplotlib


In [ ]:
import gymnasium as gym
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import matplotlib.pyplot as plt

print('Libraries loaded!')
print('PyTorch version:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 1. Q-Network

A fully-connected network: **state → Q-values for each action**

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),      nn.ReLU(),
            nn.Linear(hidden, action_size)
        )

    def forward(self, x):
        return self.net(x)

# Quick test
net = QNetwork(4, 2)
dummy = torch.randn(1, 4)
print('Output shape:', net(dummy).shape)  # (1, 2)


## 2. Replay Buffer (Experience Replay)

Stores transitions and returns random mini-batches to break temporal correlations.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=20000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            np.array(s,  dtype=np.float32),
            np.array(a,  dtype=np.int64),
            np.array(r,  dtype=np.float32),
            np.array(ns, dtype=np.float32),
            np.array(d,  dtype=np.float32),
        )

    def __len__(self):
        return len(self.buf)

print('ReplayBuffer defined!')


## 3. DQL Agent

Ties together the Q-network, target network, replay buffer, and training logic.

In [ ]:
class DQLAgent:
    def __init__(self, state_size, action_size,
                 lr=1e-3, gamma=0.99,
                 eps=1.0, eps_min=0.01, eps_decay=0.995,
                 batch_size=64, target_update=10):

        self.n_actions    = action_size
        self.gamma        = gamma
        self.eps          = eps
        self.eps_min      = eps_min
        self.eps_decay    = eps_decay
        self.batch_size   = batch_size
        self.target_update= target_update
        self.step_count   = 0
        self.device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # Networks
        self.online = QNetwork(state_size, action_size).to(self.device)
        self.target = QNetwork(state_size, action_size).to(self.device)
        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()

        self.optimizer = optim.Adam(self.online.parameters(), lr=lr)
        self.memory    = ReplayBuffer()

    # ---- Epsilon-Greedy Action Selection ----
    def act(self, state):
        if random.random() < self.eps:
            return random.randrange(self.n_actions)
        s = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self.online(s).argmax().item()

    # ---- Store Transition ----
    def remember(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    # ---- One Training Step ----
    def learn(self):
        if len(self.memory) < self.batch_size:
            return None

        s, a, r, ns, d = self.memory.sample(self.batch_size)
        s  = torch.FloatTensor(s).to(self.device)
        a  = torch.LongTensor(a).to(self.device)
        r  = torch.FloatTensor(r).to(self.device)
        ns = torch.FloatTensor(ns).to(self.device)
        d  = torch.FloatTensor(d).to(self.device)

        # Current Q-values
        q_pred = self.online(s).gather(1, a.unsqueeze(1)).squeeze(1)

        # Target Q-values (Bellman)
        with torch.no_grad():
            q_next   = self.target(ns).max(1)[0]
            q_target = r + self.gamma * q_next * (1 - d)

        loss = nn.MSELoss()(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        self.eps = max(self.eps_min, self.eps * self.eps_decay)

        # Update target network
        self.step_count += 1
        if self.step_count % self.target_update == 0:
            self.target.load_state_dict(self.online.state_dict())

        return loss.item()

print('DQLAgent defined!')


## 4. Training on CartPole-v1

**Goal:** keep the pole balanced as long as possible (max 500 steps per episode).

In [ ]:
env        = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]  # 4
n_actions  = env.action_space.n              # 2

print(f'State size : {state_size}')
print(f'Actions    : {n_actions}')

agent      = DQLAgent(state_size, n_actions)
N_EPISODES = 500
scores     = []

for ep in range(1, N_EPISODES + 1):
    state, _ = env.reset()
    total_r  = 0

    for _ in range(500):
        action              = agent.act(state)
        ns, reward, term, trunc, _ = env.step(action)
        done                = term or trunc
        agent.remember(state, action, reward, ns, done)
        agent.learn()
        state   = ns
        total_r += reward
        if done:
            break

    scores.append(total_r)

    if ep % 50 == 0:
        mean50 = np.mean(scores[-50:])
        print(f'Episode {ep:4d}  |  Avg-50: {mean50:6.1f}  |  eps: {agent.eps:.3f}')

env.close()
print('Training complete!')


## 5. Plot Results

In [ ]:
window   = 20
mov_avg  = np.convolve(scores, np.ones(window)/window, mode='valid')

plt.figure(figsize=(12, 5))
plt.plot(scores,  alpha=0.35, color='steelblue', label='Episode Score')
plt.plot(range(window-1, len(scores)), mov_avg,
         color='crimson', lw=2, label=f'{window}-ep Moving Avg')
plt.axhline(475, color='green', ls='--', lw=1.5, label='Solved (475)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Deep Q-Learning — CartPole-v1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
